# Two-Point Seismic Inverse Problem Demo

**Problem setting (Section 2.7).**  
An earthquake occurs at an unknown location $x \in \mathbb{S}^2$.  
Two sensors at $y_1, y_2 \in \mathbb{S}^2$ record signals
$$
I(x,y) = \exp\!\bigl(\beta(\langle x,y\rangle - 1)\bigr), \qquad U_i \mid x,y_i \sim \mathcal{N}(I(x,y_i),\sigma^2).
$$
Given measurements $(u_1,y_1),(u_2,y_2)$ we recover $x$ using **DPnP**.

**Sensor placement options**
- **Fixed sensors**: specify `y1`, `y2` directly.
- **vMF sensors**: set `kappa_sensors`; then $y_i \sim \mathrm{vMF}(x_{\\rm true}, \kappa_{\\rm sensors})$ so sensors are randomly placed near the earthquake.

**Reconstruction metric options**: `metric="cosine"` (dot product, ↑ better) or `metric="geodesic"` (great-circle degrees, ↓ better).

**New in this demo:**
- `n_sensor_pairs` (S): multiple $(y_1,y_2)$ configurations per $x_\\mathrm{true}$.
- `n_trials_per_sensor` (T): multiple $(u_1,u_2)$ draws per sensor pair.
- **Reflection point** $x'$: the unique point on $\mathbb{S}^2$ with identical seismic likelihood as $x_\\mathrm{true}$ (reflected through $\mathrm{span}(y_1,y_2)$), shown as a purple ✗ in all plots.
- **Baseline** `include_baseline=True`: DPnP with a flat (zero) prior, sampling $q(x) \propto p(u_1|x,y_1)\,p(u_2|x,y_2)$ without the learned earthquake distribution — dashed curves in quality plots.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from data import make_earth_dataloaders
from model import AmbientGeneratorScoreNet, DPnPScoreWrapper
from bel import get_bel
from utils import (
    normalize_torch,
    sample_vmf_s2,
    sample_xtrue_y_batches_from_dataloader,
    seismic_signal,
    get_seismic_f_fn,
    get_two_point_seismic_f_fn,
    extrinsic_to_latlon_deg_torch,
    sphere_mean_torch,
)
from test_earth import (
    run_two_point_earthquake_demo,
    cosine_similarity_vs_steps_two_point,
    plot_spherical_kde_mollweide,
    _reflect_through_great_circle,
)

device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype  = torch.float32
print(f"Using device: {device}")

## 1. Load earthquake data and trained prior score

In [ ]:
batch_size = 512
seed = 0
train_loader, val_loader, test_loader, dataset = make_earth_dataloaders(
    data_dir="data",
    name="earthquake",
    batch_size=batch_size,
    seed=seed,
    device="cpu",
)
print(f"Dataset size: {len(dataset)}")

hidden_dim      = 256
n_hidden_layers = 4
MODEL_PATH      = "p_score_3_stable.pth"   # adjust path if needed

p_model = AmbientGeneratorScoreNet(hidden_dim=hidden_dim, n_hidden_layers=n_hidden_layers)
p_model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu", weights_only=False))
p_model.eval().to(device)
p_score = DPnPScoreWrapper(p_model)
print("Prior score model loaded.")

## 2. DPnP schedule and seismic model parameters

In [ ]:
def aneal_schedule(K=20, K0=5, eta0=0.45, etaK=0.15):
    etas = np.zeros(K)
    for i in range(K):
        if i < K0:
            etas[i] = eta0
        else:
            frac = (i - K0) / (K - K0)
            etas[i] = eta0 * (etaK / eta0) ** frac
    return etas

eta = aneal_schedule(K=20, K0=5, eta0=0.1, etaK=0.05)

BETA   = 10.0    # signal decay; higher = sharper peak around y
SIGMA2 = 0.05    # observation noise variance
print(f"beta={BETA}, sigma2={SIGMA2}")

## 3. Choose x_true

Sample from the earthquake test set, or specify lat/lon directly.

In [ ]:
# --- Option A: sample from test set ---
x_true_candidates, _ = sample_xtrue_y_batches_from_dataloader(
    dataloader=test_loader, sigma_y=1.0, num_pairs=10,
    device=device, dtype=dtype, seed=42,
)
x_true = x_true_candidates[0]

# --- Option B: specify lat/lon (uncomment) ---
# import math
# lat_deg, lon_deg = 35.0, 139.0   # near Tokyo
# lat, lon = math.radians(lat_deg), math.radians(lon_deg)
# x_true = normalize_torch(torch.tensor(
#     [math.cos(lat)*math.cos(lon), math.cos(lat)*math.sin(lon), math.sin(lat)],
#     dtype=dtype, device=device))

ll = extrinsic_to_latlon_deg_torch(x_true[None]).squeeze()
print(f"x_true  (lat, lon) = ({ll[0].item():.2f}°, {ll[1].item():.2f}°)")

## 4. Demo A – fixed sensor locations

Sensors `y1`, `y2` are specified manually (random on the sphere here).

In [ ]:
torch.manual_seed(7)
y1_fixed = normalize_torch(torch.randn(3, dtype=dtype, device=device))
y2_fixed = normalize_torch(torch.randn(3, dtype=dtype, device=device))

for yi, name in [(y1_fixed, "y1"), (y2_fixed, "y2")]:
    ll = extrinsic_to_latlon_deg_torch(yi[None]).squeeze()
    print(f"{name} (lat, lon) = ({ll[0].item():.2f}°, {ll[1].item():.2f}°)")

# Reflection of x_true through span(y1, y2) — same seismic likelihood
x_refl = _reflect_through_great_circle(x_true[None], y1_fixed[None], y2_fixed[None]).squeeze()
ll_r = extrinsic_to_latlon_deg_torch(x_refl[None]).squeeze()
print(f"x' reflection (lat, lon) = ({ll_r[0].item():.2f}°, {ll_r[1].item():.2f}°)")

demo_fixed = run_two_point_earthquake_demo(
    x_true=x_true,
    y1=y1_fixed, y2=y2_fixed,
    beta=BETA, sigma2=SIGMA2,
    p_score=p_score, eta=eta,
    n_sensor_pairs=3,          # S fixed-sensor configurations (all the same here)
    n_trials_per_sensor=10,    # T u-draws per sensor pair
    metric="cosine",
    n_bel_paths=5000, n_bel_steps=5,
    out_samples=32, grw_steps=5,
    particle_counts=[1, 5, 10, 20],
    plot_per_sensor=True,      # one Mollweide per sensor pair (with x' marker)
    plot_global=True,          # global summary plot
    plot_quality_vs_steps=True,
    include_baseline=True,     # overlay likelihood-only baseline (dashed)
    device=device, dtype=dtype, seed=0,
)

## 5. Demo B – sensors sampled from vMF centred at x_true

Set `kappa_sensors` to control how close sensors are to the earthquake:
- `kappa_sensors=5`  → sensors spread broadly around x_true
- `kappa_sensors=30` → sensors tightly clustered near x_true

With `n_sensor_pairs=S`, S independent $(y_1,y_2)$ pairs are drawn from
$\mathrm{vMF}(x_{\rm true}, \kappa_{\rm sensors})$.  For each pair T observations
$(u_1,u_2)$ are drawn, giving S×T total DPnP trials.

The **reflection point** $x'_s$ (purple ✗) is computed per sensor pair — it is the
unique point whose seismic likelihood matches $x_{\rm true}$ exactly.  The prior
score pushes DPnP away from geophysically implausible $x'_s$ locations.

In [ ]:
demo_vmf = run_two_point_earthquake_demo(
    x_true=x_true,
    # y1, y2 omitted – sampled automatically from vMF
    beta=BETA, sigma2=SIGMA2,
    p_score=p_score, eta=eta,
    n_sensor_pairs=5,          # S
    n_trials_per_sensor=10,    # T
    kappa_sensors=10.0,        # sensors ~moderately close to x_true
    metric="cosine",
    n_bel_paths=5000, n_bel_steps=5,
    out_samples=32, grw_steps=5,
    particle_counts=[1, 5, 10, 20],
    plot_per_sensor=True,
    plot_global=True,
    plot_quality_vs_steps=True,
    include_baseline=True,
    device=device, dtype=dtype, seed=0,
)

# Print sampled sensor and reflection locations
for s in range(demo_vmf["y1_sensors"].shape[0]):
    ll1 = extrinsic_to_latlon_deg_torch(demo_vmf["y1_sensors"][s:s+1]).squeeze()
    ll2 = extrinsic_to_latlon_deg_torch(demo_vmf["y2_sensors"][s:s+1]).squeeze()
    llr = extrinsic_to_latlon_deg_torch(demo_vmf["x_reflected"][s:s+1]).squeeze()
    print(f"Pair {s+1}:  y1=({ll1[0]:.1f}°,{ll1[1]:.1f}°)  "
          f"y2=({ll2[0]:.1f}°,{ll2[1]:.1f}°)  "
          f"x'=({llr[0]:.1f}°,{llr[1]:.1f}°)  "
          f"score={demo_vmf['score_sensors'][s].item():.4f}")

## 6. Demo C – geodesic distance metric

Use `metric="geodesic"` to report great-circle distance in degrees (lower = better).

In [ ]:
demo_geo = run_two_point_earthquake_demo(
    x_true=x_true,
    y1=y1_fixed, y2=y2_fixed,
    beta=BETA, sigma2=SIGMA2,
    p_score=p_score, eta=eta,
    n_sensor_pairs=3,
    n_trials_per_sensor=10,
    metric="geodesic",         # ← geodesic distance in degrees (↓ better)
    n_bel_paths=5000, n_bel_steps=5,
    out_samples=32, grw_steps=5,
    particle_counts=[1, 5, 10, 20],
    plot_per_sensor=True,
    plot_global=True,
    plot_quality_vs_steps=True,
    include_baseline=True,
    device=device, dtype=dtype, seed=0,
)

## 7. Cosine similarity vs steps – fixed sensors, multiple x_true

Mirrors `cosine_similarity_vs_steps_for_particle_counts` for the two-point observation model.  
Samples many `x_true` from the earthquake test set; one $(u_1, u_2)$ pair per `x_true`.

In [ ]:
summary_fixed = cosine_similarity_vs_steps_two_point(
    dataloader=test_loader,
    p_score=p_score, eta=eta,
    y1=y1_fixed, y2=y2_fixed,
    beta=BETA, sigma2=SIGMA2,
    metric="cosine",
    particle_counts=[1, 5, 10, 20],
    out_samples=32, grw_steps=5,
    num_pairs=40, batch_eval_size=8,
    n_bel_paths=5000, n_bel_steps=5,
    include_baseline=True,     # dashed curves = likelihood-only baseline
    device=device, dtype=dtype, seed=0,
)

## 8. Cosine similarity vs steps – vMF sensors per x_true

Set `kappa_sensors` so each `x_true` gets its own $(y_1, y_2)$ drawn from $\mathrm{vMF}(x_{\rm true}, \kappa_{\rm sensors})$.

In [ ]:
summary_vmf = cosine_similarity_vs_steps_two_point(
    dataloader=test_loader,
    p_score=p_score, eta=eta,
    # y1, y2 omitted – sampled per x_true
    beta=BETA, sigma2=SIGMA2,
    kappa_sensors=10.0,        # sensors drawn from vMF centred at each x_true
    metric="cosine",
    particle_counts=[1, 5, 10, 20],
    out_samples=32, grw_steps=5,
    num_pairs=40, batch_eval_size=8,
    n_bel_paths=5000, n_bel_steps=5,
    include_baseline=True,
    device=device, dtype=dtype, seed=0,
)

## 9. Geodesic distance vs steps – vMF sensors per x_true

In [ ]:
summary_geo = cosine_similarity_vs_steps_two_point(
    dataloader=test_loader,
    p_score=p_score, eta=eta,
    kappa_sensors=10.0,
    beta=BETA, sigma2=SIGMA2,
    metric="geodesic",         # ← geodesic distance in degrees
    particle_counts=[1, 5, 10, 20],
    out_samples=32, grw_steps=5,
    num_pairs=40, batch_eval_size=8,
    n_bel_paths=5000, n_bel_steps=5,
    include_baseline=True,
    device=device, dtype=dtype, seed=0,
)

## 10. Effect of sensor concentration (kappa_sensors)

Compare reconstruction quality as sensors get closer to / farther from x_true.

In [ ]:
kappas = [2.0, 5.0, 15.0, 30.0]
final_cos_by_kappa = {}

for kappa_s in kappas:
    res = cosine_similarity_vs_steps_two_point(
        dataloader=test_loader,
        p_score=p_score, eta=eta,
        kappa_sensors=kappa_s,
        beta=BETA, sigma2=SIGMA2,
        metric="cosine",
        particle_counts=[10],
        out_samples=20, grw_steps=5,
        num_pairs=20, batch_eval_size=4,
        n_bel_paths=3000, n_bel_steps=5,
        device=device, dtype=dtype, seed=0,
    )
    final_cos = res["by_particle_count"][10]["mean_score"][-1]
    final_cos_by_kappa[kappa_s] = final_cos
    print(f"kappa_sensors={kappa_s:>5.1f}  |  final cosine = {final_cos:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(list(final_cos_by_kappa.keys()), list(final_cos_by_kappa.values()),
         marker="o", linewidth=2)
plt.xlabel(r"$\kappa_{\rm sensors}$ (vMF concentration)")
plt.ylabel("final step cosine similarity (↑ better)")
plt.title("Reconstruction quality vs sensor concentration")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. (Optional) Single-sensor vs two-sensor comparison

In [ ]:
from DPnP import dPnP_sampler_torch_batched
import math

sigma = math.sqrt(SIGMA2)
I1_true = seismic_signal(x_true, y1_fixed, BETA).item()
I2_true = seismic_signal(x_true, y2_fixed, BETA).item()

torch.manual_seed(0)
u1 = I1_true + sigma * torch.randn(1).item()
u2 = I2_true + sigma * torch.randn(1).item()

OUT_SAMPLES = 32

# ---- single sensor (y1 only) ----
q1 = get_bel(f_fn=get_seismic_f_fn(y1_fixed, u1, BETA, SIGMA2),
             n_paths=5000, n_steps=5, device=device, dtype=dtype)
X_1s = dPnP_sampler_torch_batched(
    q_score=q1, p_score=p_score, y=y1_fixed[None],
    out_samples=OUT_SAMPLES, eta=eta, grw_steps=5,
    seed=0, end_only=True, device=device, dtype=dtype,
)  # (1, P, 3)
cos_1s = (sphere_mean_torch(X_1s[0], dim=0) * x_true).sum().item()

# ---- two sensors ----
q2 = get_bel(f_fn=get_two_point_seismic_f_fn(y1_fixed, y2_fixed, u1, u2, BETA, SIGMA2),
             n_paths=5000, n_steps=5, device=device, dtype=dtype)
X_2s = dPnP_sampler_torch_batched(
    q_score=q2, p_score=p_score, y=y1_fixed[None],
    out_samples=OUT_SAMPLES, eta=eta, grw_steps=5,
    seed=0, end_only=True, device=device, dtype=dtype,
)  # (1, P, 3)
cos_2s = (sphere_mean_torch(X_2s[0], dim=0) * x_true).sum().item()

print(f"Single-sensor cosine = {cos_1s:.4f}")
print(f"Two-sensor   cosine = {cos_2s:.4f}")

for X_final, title_str in [
    (X_1s, f"Single sensor (y1) | cos={cos_1s:.4f}"),
    (X_2s, f"Two sensors (y1,y2) | cos={cos_2s:.4f}"),
]:
    plot_spherical_kde_mollweide(
        X_final=X_final,
        x_true=x_true,
        y_obs=torch.stack([y1_fixed, y2_fixed]),
        overlay_samples=True, overlay_y=True,
        title=title_str,
    )